
# 1. Dataset Understanding

## 1.1 Dataset Overview

In this section, we will inspect the raw training and testing datasets before performing any preprocessing or feature engineering.

Our objectives are to:

- Identify all available CSV files.
- Inspect the shape and column names of each file.
- Examine data types and missing values.
- Preview sample records.
- Understand the structure of the training and testing datasets.

**Important:** No data merging or modification will be performed at this stage.


In [1]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
d:\Users\PC\Desktop\To the new pc\Elzero\ML-Projects\IndabaX2025-DrySpell-Challenge\.venv\Scripts\python.exe

Python version:
3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]


In [2]:
import sys
from pathlib import Path

# Identify project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import project configuration
from config import (
    TRAIN_DIR,
    TEST_DIR,
    DATA_DIR,
    MODELS_DIR,
    REPORTS_DIR
)

print("Project root:", PROJECT_ROOT)
print("Training directory:", TRAIN_DIR)
print("Testing directory:", TEST_DIR)

Project root: D:\Users\PC\Desktop\To the new pc\Elzero\ML-Projects\IndabaX2025-DrySpell-Challenge
Training directory: D:\Users\PC\Desktop\To the new pc\Elzero\ML-Projects\IndabaX2025-DrySpell-Challenge\data\train
Testing directory: D:\Users\PC\Desktop\To the new pc\Elzero\ML-Projects\IndabaX2025-DrySpell-Challenge\data\test


In [3]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn

print("Libraries imported successfully!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"SciPy: {scipy.__version__}")

Libraries imported successfully!
NumPy: 2.5.5
Pandas: 3.1.1
Scikit-learn: 1.9.1
SciPy: 1.18.1


In [4]:
from src.data_loader import (
    inspect_file,
    build_dataset_inventory,
    analyze_date_alignment,
    build_date_availability_matrix,
    inspect_dataset_columns,
    build_rename_preview,
    load_and_standardize_datasets
)


## 1.2 Inventory of Raw Dataset Files

Before analyzing individual variables, we need to establish a complete inventory of the raw training and testing files.

For each CSV file, we will inspect:

- File name and dataset split.
- Number of rows and columns.
- Column names and data types.
- Missing values per column.
- A small preview of the records.

To avoid loading all datasets into memory simultaneously, we will process the files individually in chunks.

**Important:** This is an inspection stage only. The original files will remain unchanged.


In [5]:
# Build inventory of all raw datasets
dataset_inventory = build_dataset_inventory(
    TRAIN_DIR,
    TEST_DIR
)

# Create a compact summary
inventory_summary = pd.DataFrame([
    {
        "Dataset": item["Dataset"],
        "File": item["File"],
        "Rows": item["Rows"],
        "Columns": item["Columns"],
        "Total Missing Values": sum(
            item["Missing Values"].values()
        )
    }
    for item in dataset_inventory
])

display(inventory_summary)

,Dataset,File,Rows,Columns,Total Missing Values
0,Train,5cm soil moist CFS_train.csv,2214,2,0
1,Train,dew point_train.csv,2214,2,0
2,Train,max surface temp era5 max_train.csv,2214,2,0
3,Train,sea level pressure era5_train.csv,2214,2,0
4,Train,solution_train.csv,2214,2,0
5,Train,sst_cameroon_train.csv,2214,5,0
6,Train,sst_indian_ocean_train.csv,2214,5,0
7,Train,sudd water deficit_train.csv,2214,2,0
8,Train,surface pressure era5_train.csv,2214,2,0
9,Train,temp_trimmed_train.csv,2214,2,0



### Key Findings

The initial inspection of the raw datasets revealed the following:

1. **Dataset Files:** The training dataset contains 11 CSV files, while the testing dataset contains 10 CSV files.

2. **Training Data Consistency:** All training files contain 2,214 rows.

3. **Testing Data Variations:** Most testing files contain either 737 or 738 rows. However, both SST datasets (`sst_cameroon_test.csv` and `sst_indian_ocean_test.csv`) contain only 615 rows.

4. **Missing Values:** No missing values were detected in any of the inspected CSV files.

5. **Feature Structure:** Most files contain 2 columns, while the two SST datasets contain 5 columns each.

**Important Consideration:**

The differences in testing dataset sizes indicate that temporal alignment must be investigated before merging any files. Equal row counts or the absence of missing values do not guarantee that records correspond to the same dates.

**Next Step:** Inspect column names, data types, sample records, and date ranges to understand the structure and temporal coverage of each dataset.



## 1.3 Schema and Temporal Inspection

In this section, we will investigate the internal structure of each raw dataset.

Our objectives are to:

- Identify the exact column names in every file.
- Examine the data types.
- Preview the first five records.
- Identify potential date and time columns.
- Compare the schemas of training and testing files.

This step will help us understand how the climate variables are represented and whether the datasets can be safely aligned later.


In [6]:
# Display detailed information for each dataset
for item in dataset_inventory:

    print(f"\n{'=' * 60}")
    print(f"Dataset: {item['File']}")
    print(f"Type: {item['Dataset']}")
    print(f"Column Names: {item['Column Names']}")
    print(f"Data Types: {item['Data Types']}")
    print(f"Missing Values: {item['Missing Values']}")

    display(item["Preview"])



Dataset: 5cm soil moist CFS_train.csv
Type: Train
Column Names: ['date', '5cm_soli_moist']
Data Types: {'date': 'str', '5cm_soli_moist': 'float64'}
Missing Values: {'date': 0, '5cm_soli_moist': 0}


,date,5cm_soli_moist
0,2002-07-01,0.1391
1,2002-07-02,0.1378
2,2002-07-03,0.1371



Dataset: dew point_train.csv
Type: Train
Column Names: ['date', 'mean_dew_point_temp']
Data Types: {'date': 'str', 'mean_dew_point_temp': 'float64'}
Missing Values: {'date': 0, 'mean_dew_point_temp': 0}


,date,mean_dew_point_temp
0,2002-07-01,16.3077
1,2002-07-02,16.7714
2,2002-07-03,19.7184



Dataset: max surface temp era5 max_train.csv
Type: Train
Column Names: ['date', 'max_temp']
Data Types: {'date': 'str', 'max_temp': 'float64'}
Missing Values: {'date': 0, 'max_temp': 0}


,date,max_temp
0,2002-07-01,40.1287
1,2002-07-02,41.3576
2,2002-07-03,38.0545


... (32 outputs left out)


Dataset: temp_trimmed_test.csv
Type: Test
Column Names: ['date', '2m_temp']
Data Types: {'date': 'str', '2m_temp': 'float64'}
Missing Values: {'date': 0, '2m_temp': 0}


,date,2m_temp
0,2020-07-01,32.3152
1,2020-07-02,31.1876
2,2020-07-03,30.9356



Dataset: vapour pressure deficit_test.csv
Type: Test
Column Names: ['date', 'vapor_pressure_dificit']
Data Types: {'date': 'str', 'vapor_pressure_dificit': 'float64'}
Missing Values: {'date': 0, 'vapor_pressure_dificit': 0}


,date,vapor_pressure_dificit
0,2020-07-01,4.9632
1,2020-07-02,4.1520
2,2020-07-03,4.5619



### Key Findings

The schema inspection revealed the following:

1. **Common Date Column:** All inspected datasets use a `date` column as their temporal reference. However, it is currently stored as a string (`str`), not as a datetime data type.

2. **Numerical Features:** Climate measurements are stored as numerical `float64` columns.

3. **Target Variable:** The training target is stored in `solution_train.csv` under the column `dryspell_warn_7d`. The target file is available only in the training dataset.

4. **SST Dataset Structure:** Both SST datasets contain the same four measurement columns:
   - `mean temperature kelvin`
   - `mean temperature deg C`
   - `mean temperature uncertainty`
   - `fraction of sea-ice-covered ocean`

5. **Observed Date Samples:** The first displayed training records begin on `2002-07-01`, while the first displayed testing records begin on `2020-07-01`.

**Important Considerations:**

- The `date` column must eventually be converted to a datetime type.
- The SST temperature columns may contain redundant information; this should be investigated before feature selection.
- A common date column does not guarantee identical date coverage or unique records across files.

**Next Step:** Perform a temporal integrity check to identify date ranges, duplicate dates, missing dates, and date alignment across all files.



## 1.4 Date Range and Temporal Integrity

Since the datasets represent daily climate observations, temporal consistency is essential before combining the individual files.

In this section, we will investigate:

- Minimum and maximum dates in each file.
- Number of unique dates.
- Duplicate dates.
- Invalid date values.
- Missing days within each file's date range.
- Differences in date coverage between training and testing files.

No merging or modification of the original datasets will be performed.


In [7]:

temporal_results = []

for item in dataset_inventory:

    folder = TRAIN_DIR if item["Dataset"] == "Train" else TEST_DIR
    file_path = folder / item["File"]

    df = pd.read_csv(file_path)

    dates = pd.to_datetime(df["date"], errors="coerce")

    valid_dates = dates.dropna()
    
    if valid_dates.empty:
        temporal_results.append({
            "Dataset": item["Dataset"],
            "File": item["File"],
            "Rows": len(df),
            "Start Date": None,
            "End Date": None,
            "Unique Dates": 0,
            "Invalid Dates": dates.isna().sum(),
            "Duplicate Dates": 0,
            "Missing Days": None,
            "Sorted": False
        })
        continue

    date_range = pd.date_range(
        start=valid_dates.min(),
        end=valid_dates.max(),
        freq="D"
    )

    unique_dates = valid_dates.nunique()

    temporal_results.append({
        "Dataset": item["Dataset"],
        "File": item["File"],
        "Rows": len(df),
        "Start Date": valid_dates.min(),
        "End Date": valid_dates.max(),
        "Unique Dates": unique_dates,
        "Invalid Dates": dates.isna().sum(),
        "Duplicate Dates": valid_dates.duplicated().sum(),
        "Missing Days": len(date_range.difference(valid_dates)),
        "Sorted": dates.dropna().is_monotonic_increasing
    })


temporal_summary = pd.DataFrame(temporal_results)

display(temporal_summary)


,Dataset,File,Rows,Start Date,End Date,Unique Dates,Invalid Dates,Duplicate Dates,Missing Days,Sorted
0,Train,5cm soil moist CFS_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True
1,Train,dew point_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True
2,Train,max surface temp era5 max_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True
3,Train,sea level pressure era5_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True
4,Train,solution_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True
5,Train,sst_cameroon_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True
6,Train,sst_indian_ocean_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True
7,Train,sudd water deficit_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True
8,Train,surface pressure era5_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True
9,Train,temp_trimmed_train.csv,2214,2002-07-01,2019-10-31,2214,0,0,4118,True



### 1.4.1 Seasonal Date Coverage Analysis

The initial temporal inspection suggests that the datasets may contain seasonal observations rather than continuous daily observations throughout the year.

The expected observation window appears to be July 1 through October 31.

In this analysis, we will:

- Generate the expected dates for each observed year within the seasonal window.
- Identify missing dates within the expected observation period.
- Count duplicate and invalid dates.
- Compare date coverage across files.

Days outside the expected seasonal window will not be classified as missing observations.

No original data will be modified.


In [8]:

seasonal_results = []
file_date_sets = {}

for item in dataset_inventory:

    folder = TRAIN_DIR if item["Dataset"] == "Train" else TEST_DIR
    file_path = folder / item["File"]

    df = pd.read_csv(file_path)

    dates = pd.to_datetime(df["date"], errors="coerce")
    valid_dates = dates.dropna()

    if valid_dates.empty:
        continue

    # Generate expected seasonal dates for observed years
    expected_dates = pd.DatetimeIndex([])

    for year in range(valid_dates.min().year, valid_dates.max().year + 1):

        year_dates = pd.date_range(
            start=f"{year}-07-01",
            end=f"{year}-10-31",
            freq="D"
        )

        expected_dates = expected_dates.union(year_dates)

    actual_dates = pd.DatetimeIndex(valid_dates.unique())

    missing_dates = expected_dates.difference(actual_dates)
    unexpected_dates = actual_dates.difference(expected_dates)

    file_date_sets[item["File"]] = set(actual_dates)

    seasonal_results.append({
        "Dataset": item["Dataset"],
        "File": item["File"],
        "Rows": len(df),
        "Expected Seasonal Days": len(expected_dates),
        "Actual Unique Dates": len(actual_dates),
        "Missing Seasonal Days": len(missing_dates),
        "Duplicate Dates": valid_dates.duplicated().sum(),
        "Invalid Dates": dates.isna().sum(),
        "Unexpected Dates": len(unexpected_dates),
        "Missing Date List": [d.strftime("%Y-%m-%d") for d in missing_dates]
    })


seasonal_summary = pd.DataFrame(seasonal_results)

display(seasonal_summary)

,Dataset,File,Rows,Expected Seasonal Days,Actual Unique Dates,Missing Seasonal Days,Duplicate Dates,Invalid Dates,Unexpected Dates,Missing Date List
0,Train,5cm soil moist CFS_train.csv,2214,2214,2214,0,0,0,0,[]
1,Train,dew point_train.csv,2214,2214,2214,0,0,0,0,[]
2,Train,max surface temp era5 max_train.csv,2214,2214,2214,0,0,0,0,[]
3,Train,sea level pressure era5_train.csv,2214,2214,2214,0,0,0,0,[]
4,Train,solution_train.csv,2214,2214,2214,0,0,0,0,[]
5,Train,sst_cameroon_train.csv,2214,2214,2214,0,0,0,0,[]
6,Train,sst_indian_ocean_train.csv,2214,2214,2214,0,0,0,0,[]
7,Train,sudd water deficit_train.csv,2214,2214,2214,0,0,0,0,[]
8,Train,surface pressure era5_train.csv,2214,2214,2214,0,0,0,0,[]
9,Train,temp_trimmed_train.csv,2214,2214,2214,0,0,0,0,[]


In [9]:

# Compare all files against the date coverage of a reference file

reference_file = "5cm soil moist CFS_test.csv"

reference_dates = file_date_sets[reference_file]

date_comparison = []

for file_name, dates in file_date_sets.items():

    if file_name.endswith("_test.csv"):

        date_comparison.append({
            "File": file_name,
            "Dates in File": len(dates),
            "Dates Missing vs Reference": len(reference_dates - dates),
            "Extra Dates vs Reference": len(dates - reference_dates),
            "Same Date Coverage": dates == reference_dates
        })


date_comparison_summary = pd.DataFrame(date_comparison)

display(date_comparison_summary)


,File,Dates in File,Dates Missing vs Reference,Extra Dates vs Reference,Same Date Coverage
0,5cm soil moist CFS_test.csv,738,0,0,True
1,dew point_test.csv,737,1,0,False
2,max surface temp era5 max_test.csv,737,1,0,False
3,sea level pressure era5_test.csv,737,1,0,False
4,sst_cameroon_test.csv,615,123,0,False
5,sst_indian_ocean_test.csv,615,123,0,False
6,sudd water deficit_test.csv,737,1,0,False
7,surface pressure era5_test.csv,737,1,0,False
8,temp_trimmed_test.csv,737,1,0,False
9,vapour pressure deficit_test.csv,738,0,0,True


### Key Findings — Schema and Temporal Inspection

**1. Training Data**
- All 11 training files contain 2,214 unique dates.
- All training files are chronologically sorted.
- No duplicate, invalid, or missing seasonal dates were detected.
- The training data covers the period from July 2002 to October 2019.

**2. Testing Data**
- Most test files cover the period from July 2020 to October 2025.
- Six test files are missing the date `2023-09-11`.
- The missing date is present in the 5cm soil moisture and vapor pressure deficit files.
- No duplicate, invalid, or unexpected dates were detected.

**3. SST Data Coverage**
- Both SST test files contain 615 records.
- Their coverage ends on October 31, 2024.
- Consequently, SST observations for the 2025 test season are unavailable in the provided files.

**4. Implications**
- The missing date requires investigation before merging the datasets.
- SST coverage must be considered when designing the final feature matrix.
- An explicit date-alignment strategy is required to prevent unintended row loss or data leakage.

**Conclusion:** The training data shows consistent temporal coverage, while the test data contains date-level inconsistencies and incomplete SST coverage that must be addressed before feature engineering.

## 1.4 Cross-Dataset Date Alignment

In this section, we will investigate date alignment across all training and testing CSV files.

Although the previous inspection confirmed that individual files are chronologically sorted, it also revealed differences in their date coverage.

Our objectives are to:

- Identify the union of all available dates across datasets.
- Identify dates shared by every dataset.
- Detect dates missing from individual files.
- Compare the potential impact of `Inner Join` and `Outer Join` strategies.

This analysis will help us choose a reliable merging strategy while preserving temporal consistency and avoiding unintended data loss.

In [10]:
# Analyze date alignment across Train and Test datasets
train_alignment = analyze_date_alignment(
    TRAIN_DIR,
    "Train"
)

test_alignment = analyze_date_alignment(
    TEST_DIR,
    "Test"
)


# Display alignment results
for dataset_name, alignment in [
    ("TRAIN", train_alignment),
    ("TEST", test_alignment)
]:

    print(f"\n{'=' * 80}")
    print(f"{dataset_name} DATE ALIGNMENT")
    print(f"{'=' * 80}")

    print(
        "Total unique dates across files:",
        len(alignment["all_dates"])
    )

    print(
        "Dates shared by all files:",
        len(alignment["common_dates"])
    )

    print(
        "Dates available in at least one file:",
        len(alignment["all_dates"])
    )

    print("\nAlignment Summary:")
    display(alignment["summary"])

    print("\nDates missing from the common intersection:")

    print([
        date.strftime("%Y-%m-%d")
        for date in alignment["excluded_dates"][:20]
    ])


TRAIN DATE ALIGNMENT
Total unique dates across files: 2214
Dates shared by all files: 2214
Dates available in at least one file: 2214

Alignment Summary:


,Dataset,File,Available Dates,Missing Compared to Union,Common Dates
0,Train,5cm soil moist CFS_train.csv,2214,0,2214
1,Train,dew point_train.csv,2214,0,2214
2,Train,max surface temp era5 max_train.csv,2214,0,2214
3,Train,sea level pressure era5_train.csv,2214,0,2214
4,Train,solution_train.csv,2214,0,2214
5,Train,sst_cameroon_train.csv,2214,0,2214
6,Train,sst_indian_ocean_train.csv,2214,0,2214
7,Train,sudd water deficit_train.csv,2214,0,2214
8,Train,surface pressure era5_train.csv,2214,0,2214
9,Train,temp_trimmed_train.csv,2214,0,2214



Dates missing from the common intersection:
[]

TEST DATE ALIGNMENT
Total unique dates across files: 738
Dates shared by all files: 614
Dates available in at least one file: 738

Alignment Summary:


,Dataset,File,Available Dates,Missing Compared to Union,Common Dates
0,Test,5cm soil moist CFS_test.csv,738,0,614
1,Test,dew point_test.csv,737,1,614
2,Test,max surface temp era5 max_test.csv,737,1,614
3,Test,sea level pressure era5_test.csv,737,1,614
4,Test,sst_cameroon_test.csv,615,123,614
5,Test,sst_indian_ocean_test.csv,615,123,614
6,Test,sudd water deficit_test.csv,737,1,614
7,Test,surface pressure era5_test.csv,737,1,614
8,Test,temp_trimmed_test.csv,737,1,614
9,Test,vapour pressure deficit_test.csv,738,0,614



Dates missing from the common intersection:
['2023-09-11', '2025-07-01', '2025-07-02', '2025-07-03', '2025-07-04', '2025-07-05', '2025-07-06', '2025-07-07', '2025-07-08', '2025-07-09', '2025-07-10', '2025-07-11', '2025-07-12', '2025-07-13', '2025-07-14', '2025-07-15', '2025-07-16', '2025-07-17', '2025-07-18', '2025-07-19']



### Key Findings

**1. Training Date Alignment**
- All 11 training files share exactly 2,214 unique dates.
- No date alignment issues were detected across the training files.
- The target file (`solution_train.csv`) is aligned with all training feature files.

**2. Testing Date Alignment**
- The union of dates across all test files contains 738 unique dates.
- Only 614 dates are shared by all 10 test files.
- Six test files are missing the date `2023-09-11`.
- Both SST test files contain 615 dates and end on October 31, 2024.

**3. Main Sources of Date Misalignment**
- One missing date: `2023-09-11`, absent from six test files.
- One incomplete seasonal period: all 123 dates in the 2025 season are absent from both SST test files.

**4. Implications for Data Integration**
- An Inner Join across all test files would retain only 614 dates, potentially discarding 124 valid test records.
- The full test date coverage should be preserved during dataset construction.
- Missing feature values must be handled explicitly rather than silently removing dates.

**Conclusion:** Training datasets are fully aligned. Testing datasets require a date-preserving integration strategy to retain all 738 dates while accounting for incomplete feature coverage.



## 1.5 Date Availability Matrix

In this section, we will build a date availability matrix to visualize the temporal coverage of every dataset.

Each row represents a unique date, and each column represents a CSV file.

- 1 indicates that the date exists in the corresponding file.
- 0 indicates that the date is absent.

This matrix will help us identify the exact pattern of missing observations and design a date-preserving merging strategy.


In [11]:
# Build date availability matrices
train_availability = build_date_availability_matrix(TRAIN_DIR)
test_availability = build_date_availability_matrix(TEST_DIR)


# Display matrix dimensions
print("Train Matrix Shape:", train_availability.shape)
print("Test Matrix Shape:", test_availability.shape)


# Inspect September 2023 date availability
print("\nTest Availability: September 8–14, 2023")

display(
    test_availability.loc[
        "2023-09-08":"2023-09-14"
    ]
)


# Inspect July 2025 date availability
print("\nTest Availability: July 1–7, 2025")

display(
    test_availability.loc[
        "2025-07-01":"2025-07-07"
    ]
)

Train Matrix Shape: (2214, 11)
Test Matrix Shape: (738, 10)

Test Availability: September 8–14, 2023


,5cm soil moist CFS_test,dew point_test,max surface temp era5 max_test,sea level pressure era5_test,sst_cameroon_test,sst_indian_ocean_test,sudd water deficit_test,surface pressure era5_test,temp_trimmed_test,vapour pressure deficit_test
date,,,,,,,,,,
2023-09-08,1,1,1,1,1,1,1,1,1,1
2023-09-09,1,1,1,1,1,1,1,1,1,1
2023-09-10,1,1,1,1,1,1,1,1,1,1
2023-09-11,1,0,0,0,1,1,0,0,0,1
2023-09-12,1,1,1,1,1,1,1,1,1,1
2023-09-13,1,1,1,1,1,1,1,1,1,1
2023-09-14,1,1,1,1,1,1,1,1,1,1



Test Availability: July 1–7, 2025


,5cm soil moist CFS_test,dew point_test,max surface temp era5 max_test,sea level pressure era5_test,sst_cameroon_test,sst_indian_ocean_test,sudd water deficit_test,surface pressure era5_test,temp_trimmed_test,vapour pressure deficit_test
date,,,,,,,,,,
2025-07-01,1,1,1,1,0,0,1,1,1,1
2025-07-02,1,1,1,1,0,0,1,1,1,1
2025-07-03,1,1,1,1,0,0,1,1,1,1
2025-07-04,1,1,1,1,0,0,1,1,1,1
2025-07-05,1,1,1,1,0,0,1,1,1,1
2025-07-06,1,1,1,1,0,0,1,1,1,1
2025-07-07,1,1,1,1,0,0,1,1,1,1



### Key Findings

**1. Date Availability in the 2025 Test Season**

The date availability matrix confirms that:

- All seven inspected dates from July 1 to July 7, 2025, are available in the eight non-SST test datasets.
- Both SST datasets (`sst_cameroon_test` and `sst_indian_ocean_test`) are unavailable throughout this period.

**2. SST Coverage Limitation**

The absence of SST observations in 2025 is consistent with the previously identified end date of October 31, 2024, in both SST test files.

**3. Data Integration Implications**

- The 2025 test dates must be preserved.
- An outer date-based merge would retain these dates but produce missing SST feature values.
- The treatment of unavailable SST features requires further investigation.

**Conclusion:** The test datasets have incomplete SST temporal coverage, while the remaining climate variables are available for the inspected 2025 dates.


In [12]:
display(test_availability.loc[
    "2023-09-08":"2023-09-14"
])

,5cm soil moist CFS_test,dew point_test,max surface temp era5 max_test,sea level pressure era5_test,sst_cameroon_test,sst_indian_ocean_test,sudd water deficit_test,surface pressure era5_test,temp_trimmed_test,vapour pressure deficit_test
date,,,,,,,,,,
2023-09-08,1,1,1,1,1,1,1,1,1,1
2023-09-09,1,1,1,1,1,1,1,1,1,1
2023-09-10,1,1,1,1,1,1,1,1,1,1
2023-09-11,1,0,0,0,1,1,0,0,0,1
2023-09-12,1,1,1,1,1,1,1,1,1,1
2023-09-13,1,1,1,1,1,1,1,1,1,1
2023-09-14,1,1,1,1,1,1,1,1,1,1



### Key Findings

**1. General Date Availability**
- The date availability matrix contains 738 unique test dates and 10 feature datasets.
- The 2025 season is available in eight non-SST datasets.
- Both SST datasets do not contain observations for the 2025 season.

**2. Missing Date: September 11, 2023**
- The date `2023-09-11` is available in four datasets:
  - 5cm soil moisture
  - SST Cameroon
  - SST Indian Ocean
  - Vapor pressure deficit
- The date is absent from six datasets:
  - Dew point
  - Maximum surface temperature
  - Sea level pressure
  - Sudd water deficit
  - Surface pressure
  - Temperature

**3. Data Integration Implications**
- A date-preserving merge is necessary to retain all test dates.
- Missing feature observations must remain identifiable during integration.
- The missing observations should not be automatically interpreted as zero or replaced without investigation.

**Conclusion:** The test dataset contains two distinct temporal coverage issues: the missing date `2023-09-11` in six feature files and the absence of SST observations throughout the 2025 season.


## 1.6 Data Integration Preparation

### 1.6.1 Inspect Column Names Across Datasets

Before integrating the individual CSV files, we need to inspect their column names.

This step helps us:

- Identify columns shared across multiple datasets, especially the `date` column.
- Detect duplicated feature names across different files.
- Understand the structure of each dataset before merging.
- Identify potential naming conflicts that may occur during integration.

No data will be merged or modified in this step.

In [13]:
# Inspect columns in Train and Test datasets
train_columns = inspect_dataset_columns(
    TRAIN_DIR,
    "Train"
)

test_columns = inspect_dataset_columns(
    TEST_DIR,
    "Test"
)


# Combine column inspection results
columns_summary = pd.concat(
    [train_columns, test_columns],
    ignore_index=True
)


# Display the complete column summary
display(columns_summary)

,Dataset,File,Column,Data Type
0,Train,5cm soil moist CFS_train.csv,date,str
1,Train,5cm soil moist CFS_train.csv,5cm_soli_moist,float64
2,Train,dew point_train.csv,date,str
3,Train,dew point_train.csv,mean_dew_point_temp,float64
4,Train,max surface temp era5 max_train.csv,date,str
5,Train,max surface temp era5 max_train.csv,max_temp,float64
6,Train,sea level pressure era5_train.csv,date,str
7,Train,sea level pressure era5_train.csv,sea_level_pressure,float64
8,Train,solution_train.csv,date,str
9,Train,solution_train.csv,dryspell_warn_7d,int64


### 1.6.2 Identify Repeated Column Names

This subsection identifies column names that appear in multiple CSV files.

Repeated names are not necessarily errors. For example, the `date` column is expected to appear in every dataset.

However, repeated feature names require attention because merging datasets may create ambiguous or duplicated columns.

The goal is to distinguish expected shared columns from potential naming conflicts.

In [14]:
column_frequency = (
    columns_summary
    .groupby("Column")
    .agg(
        File_Count=("File", "nunique"),
        Files=("File", lambda x: list(x.unique()))
    )
    .reset_index()
    .sort_values("File_Count", ascending=False)
)

display(
    column_frequency[
        column_frequency["File_Count"] > 1
    ]
)

,Column,File_Count,Files
2,date,21,"[5cm soil moist CFS_train.csv, dew point_train..."
4,fraction of sea-ice-covered ocean,4,"[sst_cameroon_train.csv, sst_indian_ocean_trai..."
7,mean temperature kelvin,4,"[sst_cameroon_train.csv, sst_indian_ocean_trai..."
8,mean temperature uncertainty,4,"[sst_cameroon_train.csv, sst_indian_ocean_trai..."
6,mean temperature deg C,4,"[sst_cameroon_train.csv, sst_indian_ocean_trai..."
1,5cm_soli_moist,2,"[5cm soil moist CFS_train.csv, 5cm soil moist ..."
10,potential_water_deficit,2,"[sudd water deficit_train.csv, sudd water defi..."
5,max_temp,2,"[max surface temp era5 max_train.csv, max surf..."
0,2m_temp,2,"[temp_trimmed_train.csv, temp_trimmed_test.csv]"
9,mean_dew_point_temp,2,"[dew point_train.csv, dew point_test.csv]"


### Key Findings

- All 21 CSV files share the `date` column, making it the common key for dataset integration.
- The four SST feature names are repeated across the Cameroon and Indian Ocean datasets, in both Train and Test.
- Merging the two SST datasets without renaming their features would create ambiguous duplicate column names.
- The remaining climate features have consistent names between their corresponding Train and Test files.
- The feature `vapor_pressure_dificit` contains a spelling error that should be standardized to `vapor_pressure_deficit`.
- No columns have been renamed and no datasets have been merged at this stage.

### 1.6.3 Define a Column Renaming Strategy

Before merging the datasets, we will standardize selected feature names.

The renaming strategy includes:

1. Adding geographic identifiers to SST features to distinguish measurements from Cameroon and the Indian Ocean.
2. Correcting the spelling of `vapor_pressure_dificit` to `vapor_pressure_deficit`.
3. Keeping the `date` column unchanged because it is the common integration key.

This step defines and validates the renaming strategy without modifying the original CSV files.

In [15]:
# Generate column renaming preview

rename_preview_df = build_rename_preview(
    TRAIN_DIR,
    TEST_DIR
)

display(rename_preview_df)

,Dataset,File,Original Column,New Column
0,Train,sst_cameroon_train.csv,mean temperature kelvin,sst_cameroon_mean_temperature_kelvin
1,Train,sst_cameroon_train.csv,mean temperature deg C,sst_cameroon_mean_temperature_celsius
2,Train,sst_cameroon_train.csv,mean temperature uncertainty,sst_cameroon_mean_temperature_uncertainty
3,Train,sst_cameroon_train.csv,fraction of sea-ice-covered ocean,sst_cameroon_fraction_of_sea_ice_covered_ocean
4,Train,sst_indian_ocean_train.csv,mean temperature kelvin,sst_indian_ocean_mean_temperature_kelvin
5,Train,sst_indian_ocean_train.csv,mean temperature deg C,sst_indian_ocean_mean_temperature_celsius
6,Train,sst_indian_ocean_train.csv,mean temperature uncertainty,sst_indian_ocean_mean_temperature_uncertainty
7,Train,sst_indian_ocean_train.csv,fraction of sea-ice-covered ocean,sst_indian_ocean_fraction_of_sea_ice_covered_o...
8,Train,vapour pressure deficit_train.csv,vapor_pressure_dificit,vapor_pressure_deficit
9,Test,sst_cameroon_test.csv,mean temperature kelvin,sst_cameroon_mean_temperature_kelvin


### Key Findings

- The renaming preview contains exactly 18 changes across Train and Test datasets.
- Eight SST columns were renamed in each dataset to distinguish Cameroon and Indian Ocean measurements.
- The VPD feature name was corrected in both Train and Test datasets.
- The renaming strategy checks whether a column exists before including it in the mapping.
- Original CSV files remain unchanged.
- The renaming strategy is validated and ready to be applied during data loading.

### 1.6.4 Load and Standardize Datasets

In this subsection, we will load all CSV files into memory and apply the validated column renaming strategy.

The process will:

1. Load each CSV file independently.
2. Convert the `date` column into datetime format.
3. Apply the appropriate column renaming rules.
4. Sort each dataset by date.
5. Store the processed DataFrames in dictionaries for later integration.

The original CSV files will not be modified.

No merging, missing-value imputation, or row deletion will be performed in this step.

In [16]:
# Load Train and Test datasets
train_datasets = load_and_standardize_datasets(TRAIN_DIR)
test_datasets = load_and_standardize_datasets(TEST_DIR)

print("Train datasets loaded:", len(train_datasets))
print("Test datasets loaded:", len(test_datasets))


# Inspect standardized datasets
for dataset_name, df in train_datasets.items():

    print(f"\nDataset: {dataset_name}")
    print("Shape:", df.shape)
    print("Date dtype:", df["date"].dtype)
    print("Duplicate dates:", df["date"].duplicated().sum())
    print("Columns:", df.columns.tolist())

Train datasets loaded: 11
Test datasets loaded: 10

Dataset: 5cm soil moist CFS_train
Shape: (2214, 2)
Date dtype: datetime64[us]
Duplicate dates: 0
Columns: ['date', '5cm_soli_moist']

Dataset: dew point_train
Shape: (2214, 2)
Date dtype: datetime64[us]
Duplicate dates: 0
Columns: ['date', 'mean_dew_point_temp']

Dataset: max surface temp era5 max_train
Shape: (2214, 2)
Date dtype: datetime64[us]
Duplicate dates: 0
Columns: ['date', 'max_temp']

Dataset: sea level pressure era5_train
Shape: (2214, 2)
Date dtype: datetime64[us]
Duplicate dates: 0
Columns: ['date', 'sea_level_pressure']

Dataset: solution_train
Shape: (2214, 2)
Date dtype: datetime64[us]
Duplicate dates: 0
Columns: ['date', 'dryspell_warn_7d']

Dataset: sst_cameroon_train
Shape: (2214, 5)
Date dtype: datetime64[us]
Duplicate dates: 0
Columns: ['date', 'sst_cameroon_mean_temperature_kelvin', 'sst_cameroon_mean_temperature_celsius', 'sst_cameroon_mean_temperature_uncertainty', 'sst_cameroon_fraction_of_sea_ice_covered_oce